# Proyek Capstone: AI-Powered Interview Assessment System
**Tim A25-CS358**

- **Muhammad Rayhan**, M262D5Y1357, sebagai PIC Model & Training (Streamlit/Interface)
- **Hafiz Putra Mahesta**, M262D5Y0714, sebagai PIC Integrasi,Model STT, & Fitur (Confidence Score)
- **Fahri Rasyidin**, M262D5Y0566, sebagai PIC Data & Evaluasi (Dataset, Kunci Jawaban, WER)

**Versi Proposal Kompetisi:** core method diganti menjadi **Wav2Vec2** (STT) dan **Zephyr-7B / Mistral-7B-Instruct** (LLM Scoring). Alur data, dataset, dan evaluasi (WER/Akurasi) tidak diubah.

# Environment Setup & Dependencies

In [1]:
# [PERUBAHAN] Whisper dihapus; ditambah bitsandbytes (kuantisasi 4-bit) untuk LLM 7B.
# moviepy dipin ke 1.0.3 agar `import moviepy.editor` & `write_audiofile(verbose=...)`
# pada fungsi convert_video_to_audio tetap berjalan persis seperti versi lama
# (moviepy 2.x menghapus modul `moviepy.editor` dan argumen `verbose`).
!pip install -q transformers accelerate bitsandbytes
!pip install -q librosa soundfile jiwer moviepy==1.0.3

In [2]:
# [PERUBAHAN] `import whisper` dihapus
import os
import json
import time
import shutil
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import moviepy.editor as mp
import librosa
import soundfile as sf
import torch
import jiwer
from datetime import datetime
from tqdm.notebook import tqdm
from transformers import pipeline, BitsAndBytesConfig  # [PERUBAHAN] + BitsAndBytesConfig
import gc
from google.colab import drive

/usr/local/lib/python3.13/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':

  from pkg_resources import resource_stream, resource_exists



In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


# System Configuration & Model Initialization

In [4]:
BASE_DIR = "/content/drive/MyDrive/Dataset"
VIDEO_INPUT_DIR = os.path.join(BASE_DIR, "Video")
AUDIO_OUTPUT_DIR = os.path.join(BASE_DIR, "Audio")
GROUND_TRUTH_FILE = os.path.join(BASE_DIR, "Transkrip_Manual")

# Cek Folder Video
if os.path.exists(VIDEO_INPUT_DIR):
    video_files = [f for f in os.listdir(VIDEO_INPUT_DIR) if f.lower().endswith(('.mp4', '.webm', '.avi', '.mov', '.mkv'))]
    print(f" Folder Video ditemukan.")
    print(f" Jumlah video yang siap diproses: {len(video_files)} file")
else:
    print(f" Folder Video TIDAK ditemukan di: {VIDEO_INPUT_DIR}")

if os.path.exists(GROUND_TRUTH_FILE):
    print(f"File Transkrip Manual ditemukan.")
else:
    print(f"File Transkrip Manual tidak ditemukan di: {GROUND_TRUTH_FILE}")

# ============================================================
# [PERUBAHAN METODE AI #1] STT: Whisper("small")  ->  Wav2Vec2 (facebook/wav2vec2-base-960h)
# ============================================================
ASR_MODEL_ID = "facebook/wav2vec2-base-960h"
TARGET_SR = 16000  # Wav2Vec2 wajib input 16 kHz

try:
    asr_device = 0 if torch.cuda.is_available() else -1
    asr_pipeline = pipeline(
        "automatic-speech-recognition",
        model=ASR_MODEL_ID,
        device=asr_device,
    )
    print(f"Model Wav2Vec2 ({ASR_MODEL_ID}) berhasil dimuat pada device: {asr_device}")
except Exception as e:
    print(f" Gagal memuat model. Detail error: {e}")

 Folder Video ditemukan.
 Jumlah video yang siap diproses: 15 file
File Transkrip Manual ditemukan.


config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

Model Wav2Vec2 (facebook/wav2vec2-base-960h) berhasil dimuat pada device: 0


## Core Utility Functions

In [5]:
# [TIDAK BERUBAH] convert_video_to_audio, remove_fillers, calculate_metrics
# [PERUBAHAN] hanya transcribe_audio
def convert_video_to_audio(video_path, audio_path):
    try:
        video_clip = mp.VideoFileClip(video_path)
        video_clip.audio.write_audiofile(audio_path, codec='pcm_s16le', verbose=False, logger=None)
        video_clip.close()
        return True
    except Exception:
        return False

# ------------------------------------------------------------
# [PERUBAHAN METODE AI #1] transcribe_audio: Whisper -> Wav2Vec2
#  - Audio di-load & di-resample ke 16 kHz mono dengan librosa
#  - Inferensi CTC Wav2Vec2 dengan chunking 30 dtk (stride 5 dtk) agar
#    jawaban wawancara yang panjang tidak membuat GPU kehabisan memori
#  - `initial_prompt`/`language` milik Whisper tidak ada di Wav2Vec2
#    (model hanya English, vocab huruf kapital tanpa tanda baca)
# ------------------------------------------------------------
def transcribe_audio(audio_path):
    try:
        # librosa otomatis resample ke TARGET_SR (16 kHz) dan mengubah ke mono
        speech, _ = librosa.load(audio_path, sr=TARGET_SR, mono=True)

        result = asr_pipeline(
            {"raw": speech, "sampling_rate": TARGET_SR},
            chunk_length_s=30,
            stride_length_s=5,
        )
        # Output Wav2Vec2-960h berupa HURUF KAPITAL -> dijadikan lowercase
        return result["text"].lower().strip()
    except Exception as e:
        print(f" Transkripsi Error: {e}")
        return ""

def remove_fillers(text):
    # Daftar kata filler yang mau dihapus (bisa ditambah)
    fillers = [
        r"\bum\b", r"\buh\b", r"\buhh\b", r"\bah\b", r"\ber\b", r"\bhmm\b",
        r"\bmhm\b", r"\buh-huh\b", r"\bokay\b",
        r"\byou know\b", r"\bi mean\b", r"\bkind of\b", r"\bsort of\b",
        r"\bso\b", r"\blike\b", r"\byeah\b", r"\bright\b",
    ]

    clean_text = text.lower()
    for filler in fillers:
        clean_text = re.sub(filler, "", clean_text)

    # Hapus spasi ganda sisa penghapusan
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()
    return clean_text

def calculate_metrics(reference_text, hypothesis_text):
    if not reference_text or not hypothesis_text:
        return {"wer": 1.0, "accuracy": 0.0}

    # Bersihkan tanda baca dasar
    transformation = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.RemoveMultipleSpaces(),
        jiwer.Strip(),
    ])

    ref_basic = transformation(reference_text)
    hyp_basic = transformation(hypothesis_text)

    # Hapus filler words
    ref_clean = remove_fillers(ref_basic)
    hyp_clean = remove_fillers(hyp_basic)

    # Hitung Akurasi
    wer_score = jiwer.wer(ref_clean, hyp_clean)
    accuracy = max(0, 1 - wer_score) * 100

    return {"wer": wer_score, "accuracy": round(accuracy, 2)}

## Data Processing Pipeline (ETL)

In [6]:
# [TIDAK BERUBAH] Alur ETL & evaluasi WER sama seperti versi lama
TRANSCRIPT_DIR = os.path.join(BASE_DIR, "Transkrip_Manual")

# Ekstraksi Audio (Video -> Audio)
video_files = [f for f in os.listdir(VIDEO_INPUT_DIR) if f.lower().endswith(('.mp4', '.avi', '.webm'))]
print(f"Memulai pemrosesan untuk {len(video_files)} file video...")

for video in tqdm(video_files, desc="Converting Videos"):
    v_path = os.path.join(VIDEO_INPUT_DIR, video)
    a_path = os.path.join(AUDIO_OUTPUT_DIR, os.path.splitext(video)[0] + ".wav")

    if not os.path.exists(a_path):
        convert_video_to_audio(v_path, a_path)

# Mengumpulkan semua file audio dari hasil konversi maupun yang sudah ada di folder
all_audio_files = []

for f in os.listdir(AUDIO_OUTPUT_DIR):
    if f.lower().endswith('.wav'):
        all_audio_files.append(os.path.join(AUDIO_OUTPUT_DIR, f))

# Menghapus duplikasi path jika ada
all_audio_files = list(set(all_audio_files))
print(f"Total file audio yang siap diproses: {len(all_audio_files)} file")

# Proses Transkripsi & Evaluasi
processing_results = []
total_accuracy = 0
count_evaluated = 0

for audio_path in tqdm(all_audio_files, desc="AI Transcribing"):
    filename = os.path.basename(audio_path)
    base_name = os.path.splitext(filename)[0]

    pred_text = transcribe_audio(audio_path)

    metrics = {"accuracy": 0.0}
    truth_text = "N/A"

    txt_path = os.path.join(TRANSCRIPT_DIR, base_name + ".txt")

    if os.path.exists(txt_path):
        try:
            with open(txt_path, 'r', encoding='utf-8') as f:
                truth_text = f.read().strip()
            # Hitung akurasi
            metrics = calculate_metrics(truth_text, pred_text)
            total_accuracy += metrics["accuracy"]
            count_evaluated += 1
        except: pass

    # Simpan hasil ke list
    processing_results.append({
        "filename": filename,
        "prediction": pred_text,
        "ground_truth": truth_text[:100],
        "accuracy": metrics["accuracy"]
    })

# Laporan Ringkasan
print("-" * 50)
df_results = pd.DataFrame(processing_results)

if count_evaluated > 0:
    avg_accuracy = total_accuracy / count_evaluated
else:
    avg_accuracy = 0.0

# Tampilkan statistik akhir
print(f"Total Data Diproses       : {len(processing_results)}")
print(f"Data dengan Kunci Jawaban : {count_evaluated}")
print(f"Rata-rata Akurasi         : {avg_accuracy:.2f}%")

# Validasi terhadap target keberhasilan proyek
if avg_accuracy >= 90:
    print("STATUS: TERCAPAI")
else:
    print("STATUS: BELUM TERCAPAI")

df_results[["filename", "accuracy"]]

Memulai pemrosesan untuk 15 file video...


Converting Videos:   0%|          | 0/15 [00:00<?, ?it/s]

Total file audio yang siap diproses: 20 file


AI Transcribing:   0%|          | 0/20 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


--------------------------------------------------
Total Data Diproses       : 20
Data dengan Kunci Jawaban : 20
Rata-rata Akurasi         : 28.69%
STATUS: BELUM TERCAPAI


,filename,accuracy
0,interview_question_9.wav,36.36
1,interview_question_12.wav,0.00
2,interview_question_16.wav,33.64
3,interview_question_14.wav,14.71
4,interview_question_20.wav,52.99
5,interview_question_19.wav,42.33
6,interview_question_17.wav,53.95
7,interview_question_7.wav,16.51
8,interview_question_6.wav,26.62
9,interview_question_13.wav,16.22


# AI-Powered Assessment Engine (LLM Reasoning) dan Final Reporting & Export

In [7]:
# ============================================================
# [PERUBAHAN METODE AI #2] LLM: google/flan-t5-large (text2text)
#                       ->  Zephyr-7B-beta / Mistral-7B-Instruct (text-generation)
# ============================================================
# Catatan memori (Colab T4 = ~15 GB VRAM):
#  - 7B dalam bfloat16/float16 murni = ~14.5 GB -> nyaris penuh & rawan OOM.
#  - Solusi: bobot dikuantisasi 4-bit (NF4) via bitsandbytes -> sekitar 4-5 GB.
#  - T4 (compute capability 7.5) tidak punya dukungan native bfloat16, jadi dtype
#    komputasi otomatis: bfloat16 di GPU Ampere+ (A100/L4), float16 di T4.
LLM_MODEL_ID = "HuggingFaceH4/zephyr-7b-beta"
# Alternatif: "mistralai/Mistral-7B-Instruct-v0.2" (mungkin perlu login HF + terima lisensi)

if not torch.cuda.is_available():
    raise RuntimeError("LLM 7B membutuhkan GPU. Aktifkan: Runtime > Change runtime type > T4 GPU.")

# Bebaskan memori GPU dari model STT (transkripsi sudah selesai di sel ETL)
if 'asr_pipeline' in globals():
    del asr_pipeline
gc.collect()
torch.cuda.empty_cache()

compute_dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

assessor_llm = pipeline(
    "text-generation",
    model=LLM_MODEL_ID,
    torch_dtype=compute_dtype,
    device_map="auto",
    model_kwargs={"quantization_config": bnb_config},
)
print(f"LLM {LLM_MODEL_ID} berhasil dimuat (4-bit, compute dtype: {compute_dtype})")

config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

... (4 outputs left out)

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

LLM HuggingFaceH4/zephyr-7b-beta berhasil dimuat (4-bit, compute dtype: torch.float16)


In [8]:
# [TIDAK BERUBAH] Basis pengetahuan pertanyaan
# Basis Pengetahuan Pertanyaan
# Digunakan untuk memberikan konteks kepada AI saat menilai jawaban
QUESTION_DB = {
    1: "Share specific challenges you faced in certification and how you overcame them.",
    2: "Describe your experience with transfer learning in TensorFlow.",
    3: "Describe a complex TensorFlow model you built and how you ensured accuracy.",
    4: "Explain how to implement dropout and its effect on training.",
    5: "Describe the process of building a CNN for image classification.",
    6: "Tell us about yourself and why you want to become an AI Engineer.",
    7: "What AI/ML tools or skills have you learned so far?",
    8: "Explain what Machine Learning is in simple words.",
    9: "How do you solve problems when your model does not work well?",
    10: "What personal qualities do you have to support you as an AI Engineer?",
    11: "Why did you choose architecture as your field?",
    12: "What architectural software are you familiar with?",
    13: "How do you approach designing a building or space?",
    14: "How do you handle design criticism or multiple revisions?",
    15: "Why should we select you for this position?",
    16: "Walk me through your resume and tell me about yourself.",
    17: "Tell me about the most challenging technical problem you've faced.",
    18: "Why are you specifically interested in our company (Traveloka)?",
    19: "Tell me about a time you had a significant disagreement with a colleague.",
    20: "Explain Transfer Learning to a non-technical person."
}

In [9]:
# ------------------------------------------------------------
# [PERUBAHAN METODE AI #2] assess_with_llm: Flan-T5 -> Zephyr/Mistral
#  - Memakai chat template bawaan model (instruction-tuned)
#  - Seluruh instruksi ditaruh di 1 pesan "user" agar kompatibel dengan
#    Zephyr maupun Mistral-Instruct (Mistral v0.2 tidak mendukung role "system")
#  - Decoding greedy (do_sample=False) supaya skor deterministik/reproducible
# ------------------------------------------------------------
def assess_with_llm(answer_text, question_id):
    # Validasi jawaban yang terlalu pendek
    if not answer_text or len(answer_text) < 10:
        return 0, "Candidate did not provide a meaningful answer."

    question_text = QUESTION_DB.get(question_id, "General interview question")

    # Instruksi (Prompt) untuk LLM agar bertindak sebagai Recruiter
    prompt = f"""You are a Senior Technical Recruiter. Evaluate this interview answer.

Question: "{question_text}"
Candidate Answer: "{answer_text}"

Task:
1. Give a Score (1-4) based on depth, clarity, and technical correctness.
2. Write a short Reason (max 1 sentence).

Rubric:
1: Poor/Irrelevant.
2: Basic/General.
3: Good/Specific.
4: Excellent/Detailed.

Output Format: Score | Reason
Example: 3 | The candidate gave a relevant answer with some specific details.
Respond with exactly one line in this format and nothing else."""

    try:
        messages = [{"role": "user", "content": prompt}]
        chat_prompt = assessor_llm.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        # Menjalankan inferensi model
        output = assessor_llm(
            chat_prompt,
            max_new_tokens=80,
            do_sample=False,
            return_full_text=False,
        )[0]["generated_text"].strip()

        # Memisahkan skor dan alasan dari teks output model
        if "|" in output:
            parts = output.split("|", 1)
            score_match = re.search(r"[1-4]", parts[0])
            score = int(score_match.group()) if score_match else 2
            reason = parts[1].strip().split("\n")[0].strip()
        else:
            score_match = re.search(r"[1-4]", output)
            score = int(score_match.group()) if score_match else 3
            reason = output.split("\n")[0].strip()

        return max(1, min(4, score)), reason

    except Exception as e:
        print(f"Gagal menilai pertanyaan ID {question_id}: {e}")
        return 2, "AI could not process this specific answer."

In [10]:
# [TIDAK BERUBAH] struktur kamus generate_final_report (hanya teks 'notes' disesuaikan)
def extract_id_from_filename(filename):
    numbers = re.findall(r'\d+', filename)
    return int(numbers[0]) if numbers else 999

# Pembuatan Laporan Akhir
def generate_final_report(results_data):
    final_output = {
        "success": True,
        "data": {
            "id": 131,
            "candidate": {
                "name": "Hafiz Putra Mahesta",
                "email": "hafiz123@gmail.com",
                "photoUrl": "https://drive.google.com/file/d/1AGt7RzxoY21DlqTLBqrB74DFvSiTEDmL/view?usp=sharing"
            },
            "certification": {
                "abbreviatedType": "DCML",
                "normalType": "DEV_CERTIFICATION_MACHINE_LEARNING",
                "submittedAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "status": "FINISHED",
                "projectType": "dcml_package_1",
                "interviewQuestionSets": 1,
                "examScore": 0,
                "autoGraderProjectScore": 100,
                "downloadProjectUrl": "https://github.com/hayeee14/project-ML-ASAH-FT",
                "isReviewedByMe": False,
                "isAlreadyReviewedByMe": False,
                "assess": {
                    "project": False,
                    "interviews": True
                }
            },
            "reviewChecklists": {
                "project": [],
                "interviews": []
            },
            "pastReviews": []
        }
    }

    # Objek Reviewer
    current_review = {
        "assessorProfile": {
            "id": 47,
            "name": "AI Assessor System",
            "photoUrl": "xxx"
        },
        "decision": "PENDING",
        "reviewedAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "scoresOverview": {"project": 100, "interview": 0, "total": 0},
        "reviewChecklistResult": {
            "project": [],
            "interviews": {
                "minScore": 0,
                "maxScore": 4,
                "scores": []
            }
        },
        "notes": "Automated assessment utilizing Facebook Wav2Vec2 for transcription and Zephyr-7B (Mistral-based LLM) for reasoning."  # [PERUBAHAN] hanya teks nama model
    }

    total_score = 0
    count = 0

    # Mengurutkan data berdasarkan ID pertanyaan (1-20)
    sorted_results = sorted(results_data, key=lambda x: extract_id_from_filename(x['filename']))

    print(f"Memulai penilaian otomatis untuk {len(sorted_results)} jawaban...")

    for item in tqdm(sorted_results, desc="Penilaian AI"):
        filename = item['filename']
        q_id = extract_id_from_filename(filename)

        # Menilai Jawaban
        score, reason = assess_with_llm(item['prediction'], q_id)

        video_url = f"https://drive.google.com/file/d/{filename}/view?usp=sharing"

        # Masukkan ke Checklist
        checklist_entry = {
            "positionId": q_id,
            "question": QUESTION_DB.get(q_id, "Question text not found"),
            "isVideoExist": True,
            "recordedVideoUrl": video_url
        }
        final_output["data"]["reviewChecklists"]["interviews"].append(checklist_entry)

        # Masukkan ke Hasil Score
        score_entry = {
            "id": q_id,
            "score": score,
            "reason": reason, # Alasan dari LLM
            "transcript_preview": item['prediction'][:200] + "..." # Cuplikan transkrip
        }
        current_review["reviewChecklistResult"]["interviews"]["scores"].append(score_entry)

        if 1 <= q_id <= 20:
            total_score += score
            count += 1

    # --- PERHITUNGAN SKOR AKHIR ---
    if count > 0:
        interview_score = (total_score / (count * 4)) * 100
    else:
        interview_score = 0

    # Update Skor di JSON
    current_review["scoresOverview"]["interview"] = round(interview_score, 2)

    project_score = 100
    total_final = (project_score + interview_score) / 2
    current_review["scoresOverview"]["total"] = round(total_final, 2)

    # Update Skor Exam Utama
    final_output["data"]["certification"]["examScore"] = round(total_final, 2)

    current_review["decision"] = "PASSED" if total_final >= 75 else "REVIEW NEEDED"

    # Masukkan Review ke Data Utama
    final_output["data"]["pastReviews"].append(current_review)

    return final_output

In [11]:
# [TIDAK BERUBAH] Ekspor JSON
if 'processing_results' in locals() and processing_results:
    json_report = generate_final_report(processing_results)

    output_path = os.path.join(BASE_DIR, "final_assessment_result.json")
    with open(output_path, "w") as f:
        json.dump(json_report, f, indent=2)

    print(f"Laporan JSON berhasil disimpan: {output_path}")
else:
    print("Data transkripsi belum siap. Jalankan cell sebelumnya dulu.")

Memulai penilaian otomatis untuk 20 jawaban...


Penilaian AI:   0%|          | 0/20 [00:00<?, ?it/s]

Laporan JSON berhasil disimpan: /content/drive/MyDrive/Dataset/final_assessment_result.json
